# **1. Perkenalan Dataset**


Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   Dataset dapat diperoleh dari berbagai sumber, seperti public repositories (*Kaggle*, *UCI ML Repository*, *Open Data*) atau data primer yang Anda kumpulkan sendiri.

**Dataset yang Digunakan**: Give Me Some Credit (Credit Scoring Dataset)  
**Target**: `SeriousDlqin2yrs` (Prediksi risiko gagal bayar kredit 1/0)  
**Jumlah Data**: 150,000 sampel

# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [ ]:
import pandas as pd
import numpy as np
import os

print("Library berhasil di-import!")

# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.

Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

In [ ]:
# Path dataset raw
raw_path = '../credit_scoring_raw/cs-training.csv'
if not os.path.exists(raw_path):
    raw_path = 'credit_scoring_raw/cs-training.csv'

df = pd.read_csv(raw_path)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print(f"Ukuran dataset raw: {df.shape}")
df.head()

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [ ]:
# Ringkasan Informasi Data
print("--- Informasi Dataset ---")
print(df.info())

print("\n--- Statistik Deskriptif ---")
display(df.describe())

print("\n--- Distribusi Target (SeriousDlqin2yrs) ---")
print(df['SeriousDlqin2yrs'].value_counts(normalize=True))

print("\n--- Jumlah Missing Values ---")
print(df.isnull().sum())

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

In [ ]:
# 5.1 Handling Missing Values
df_clean = df.copy()
median_income = df_clean['MonthlyIncome'].median()
df_clean['MonthlyIncome'] = df_clean['MonthlyIncome'].fillna(median_income)
mode_dependents = df_clean['NumberOfDependents'].mode()[0]
df_clean['NumberOfDependents'] = df_clean['NumberOfDependents'].fillna(mode_dependents)

# 5.2 Handling Outliers
outlier_cols = ['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio']
for col in outlier_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 3 * IQR
    lower_bound = Q1 - 3 * IQR
    df_clean[col] = np.clip(df_clean[col], lower_bound, upper_bound)

# 5.3 Feature Engineering
df_clean['TotalLatePayments'] = (
    df_clean['NumberOfTime30-59DaysPastDueNotWorse'] +
    df_clean['NumberOfTime60-89DaysPastDueNotWorse'] +
    df_clean['NumberOfTimes90DaysLate']
)
df_clean['IncomePerDependent'] = df_clean['MonthlyIncome'] / (df_clean['NumberOfDependents'] + 1)

# 5.4 Export Dataset Preprocessing
output_dir = 'preprocessing'
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, 'credit_scoring_preprocessing.csv')
df_clean.to_csv(output_file, index=False)
print(f"Data preprocessed berhasil disimpan ke {output_file}")